# SARSA semi-gradiente en LunarLander-v3 (Gymnasium)

Este notebook adapta el enfoque SARSA tabular a **SARSA semi-gradiente** usando una red neuronal (PyTorch) como aproximador de \(\hat q(s,a;w)\).

> Si ejecutas esto en Colab y te falta Box2D, descomenta la celda de instalación.


In [1]:
# (Solo si es necesario en Colab)
# !pip -q install "gymnasium[box2d]"


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

from agents.AgentSARSASemiGrad import AgentSarsaSemiGrad

ImportError: cannot import name 'AgentSarsaSemiGrad' from 'agents.AgentSARSASemigradient' (c:\Users\USUARIO\Desktop\EML\P1\aprendizaje-por-refuerzo-EML\parte2\agents\AgentSARSASemigradient.py)

In [ ]:
def moving_average(x, window=100):
    x = np.asarray(x, dtype=np.float64)
    if len(x) < window:
        return x
    c = np.cumsum(np.insert(x, 0, 0))
    return (c[window:] - c[:-window]) / window

def plot_returns(returns, window=100):
    plt.figure()
    plt.plot(returns, label="Return por episodio")
    ma = moving_average(returns, window=window)
    if len(ma) > 0:
        plt.plot(np.arange(window-1, window-1+len(ma)), ma, label=f"Media móvil ({window})")
    plt.title("LunarLander SARSA semi-gradiente: retorno por episodio")
    plt.xlabel("Episodio")
    plt.ylabel("Return")
    plt.legend()
    plt.show()

def plot_episode_length(lengths, window=100):
    plt.figure()
    plt.plot(lengths, label="Longitud por episodio")
    ma = moving_average(lengths, window=window)
    if len(ma) > 0:
        plt.plot(np.arange(window-1, window-1+len(ma)), ma, label=f"Media móvil ({window})")
    plt.title("LunarLander SARSA semi-gradiente: longitud del episodio")
    plt.xlabel("Episodio")
    plt.ylabel("Pasos")
    plt.legend()
    plt.show()

def plot_success_ratio(successes, window=100):
    successes = np.asarray(successes, dtype=np.float64)
    plt.figure()
    plt.plot(successes, label="Éxito (0/1)")
    ma = moving_average(successes, window=window)
    if len(ma) > 0:
        plt.plot(np.arange(window-1, window-1+len(ma)), ma, label=f"Tasa éxito media móvil ({window})")
    plt.title("LunarLander SARSA semi-gradiente: tasa de éxito")
    plt.xlabel("Episodio")
    plt.ylabel("Éxito")
    plt.legend()
    plt.show()


In [ ]:
env = gym.make("LunarLander-v3")

agent = AgentSarsaSemiGrad(
    env,
    epsilon=1.0,
    decay=True,
    discount_factor=0.99,
    alpha=1e-3,
    epsilon_min=0.05,
    epsilon_decay=0.999,
    success_threshold=100.0,
    normalize_obs=True
)

n_episodes = 4000  # ajusta según tiempo
print("Device:", agent.device)

for episode in tqdm(range(n_episodes)):
    obs, info = env.reset()
    action = agent.get_action(obs)

    done = False
    while not done:
        next_obs, reward, terminated, truncated, info = env.step(action)
        next_action = agent.update(obs, action, next_obs, reward, terminated, truncated, info)
        done = terminated or truncated
        obs = next_obs
        if not done:
            action = next_action

    if (episode + 1) % 500 == 0:
        last100 = agent.stats()["returns"][-100:]
        avg = float(np.mean(last100)) if len(last100) > 0 else float("nan")
        print(f"Episodio {episode+1}: avg_return(last100)={avg:.2f}, epsilon={agent.epsilon:.4f}")

env.close()

stats = agent.stats()
returns = stats["returns"]
lengths = stats["lengths"]
successes = stats["successes"]

len(returns), len(lengths), len(successes)


In [ ]:
plot_returns(returns, window=100)
plot_episode_length(lengths, window=100)
plot_success_ratio(successes, window=100)
